# ==========================================================
# 04_sql_transformations.ipynb - Aggregations & Plan Comparison
# ==========================================================

In [ ]:
import pathlib
import pandas as pd

PROJECT_ROOT = pathlib.Path.cwd().parent
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
RAW_DIR = PROJECT_ROOT / "data" / "raw"

In [ ]:
# ----------------------------------------------------------
# Step 1: Read Data
# ----------------------------------------------------------
df_bookings = pd.read_csv(CLEAN_DIR / "bookings_clean.csv.gz", parse_dates=["slot_at"])
df_plan_raw = pd.read_csv(RAW_DIR / "plan" / "plan_2026_H1.csv")

# Extract Month string YYYY-MM
df_bookings["month"] = df_bookings["slot_at"].dt.strftime("%Y-%m")

# Reshape Plan from Wide to Long
df_plan = df_plan_raw.melt(id_vars=["city", "metric"], var_name="month", value_name="plan_2026")
df_plan["city"] = df_plan["city"].replace({"Bangalore": "Bengaluru"})
df_plan = df_plan[["city", "month", "plan_2026"]]

In [ ]:
# ----------------------------------------------------------
# Step 2: Compute Actual Test Counts (2025 vs 2026)
# ----------------------------------------------------------
df_counts = df_bookings.groupby(["city", "month"]).size().rename("actual_tests").reset_index()

# Extract 2025 actuals and map month string to 2026 for alignment
df_actual_2025 = df_counts[df_counts["month"].str.startswith("2025")].copy()
df_actual_2025["month"] = df_actual_2025["month"].str.replace("2025", "2026", regex=False)
df_actual_2025 = df_actual_2025.rename(columns={"actual_tests": "actual_2025"})

# Extract 2026 actuals
df_actual_2026 = df_counts[df_counts["month"].str.startswith("2026")].rename(columns={"actual_tests": "actual_2026"})

# Combine Plan, 2025 Actuals, and 2026 Actuals
df_comparison = df_plan.merge(df_actual_2025, on=["city", "month"], how="left") \
                       .merge(df_actual_2026, on=["city", "month"], how="left")

In [ ]:
# ----------------------------------------------------------
# Step 3: Compute Growth & Target Variance
# ----------------------------------------------------------
df_comparison["growth_pct"] = ((df_comparison["actual_2026"] / df_comparison["actual_2025"]) - 1) * 100
df_comparison["vs_plan_pct"] = ((df_comparison["actual_2026"] / df_comparison["plan_2026"]) - 1) * 100

print("--- City-wise Monthly Comparison Table ---")
print(df_comparison.head(12))

In [ ]:
# Calculate Half-Year (H1) Aggregated Totals
df_h1 = df_comparison.groupby("city")[["actual_2025", "actual_2026", "plan_2026"]].sum()
df_h1["growth_pct"] = ((df_h1["actual_2026"] / df_h1["actual_2025"]) - 1) * 100
df_h1["gap_points"] = (df_h1["actual_2026"] - df_h1["plan_2026"]) / df_h1["actual_2025"].sum() * 100

print("\n--- H1 Summary by City ---")
print(df_h1.round(2))